# Model Comparison — Logistic Regression vs Random Forest vs LightGBM
Trains all three models on the same 10K sample and compares accuracy, F1, and ROC-AUC.
Used for thesis Phase 3 evaluation.

In [1]:
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 130

from sklearn.linear_model  import LogisticRegression
from sklearn.ensemble       import RandomForestClassifier
from lightgbm               import LGBMClassifier
from sklearn.preprocessing  import OrdinalEncoder, label_binarize
from sklearn.model_selection import train_test_split
from sklearn.metrics        import (
    classification_report, accuracy_score,
    roc_auc_score, roc_curve, auc
)

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
print('Imports OK')

Imports OK


In [2]:
# Load preprocessed sample
df = pd.read_parquet('Data/processed_flights_sample.parquet')
print(f'Loaded {len(df):,} rows')
print('Class distribution:')
print(df['DELAY_CLASS'].value_counts().sort_index())

Loaded 10,000 rows
Class distribution:
DELAY_CLASS
0    8145
1    1028
2     827
Name: count, dtype: int64


In [3]:
# Load preprocessor artifacts (same pipeline as the app)
pre_info   = joblib.load('models/preprocessor_sample.joblib')
enc        = joblib.load('models/ordinal_encoder.joblib')
route_data = joblib.load('models/route_avg_delay.joblib')

FEATURE_COLS = pre_info['feature_cols']
CAT_COLS     = pre_info['cat_cols']
NUM_COLS     = pre_info['num_cols']
NUM_MEDIANS  = pre_info['num_medians']
ROUTE_AVG    = route_data['route_avg']
GLOBAL_AVG   = route_data['global_avg']

# Add route_avg_delay
df = df.merge(ROUTE_AVG[['origin_topK','dest_topK','route_avg_delay']],
              on=['origin_topK','dest_topK'], how='left')
df['route_avg_delay'] = df['route_avg_delay'].fillna(GLOBAL_AVG)

# Encode
X = df[FEATURE_COLS].copy()
y = df['DELAY_CLASS']

for col in NUM_COLS:
    X[col] = pd.to_numeric(X[col], errors='coerce').fillna(NUM_MEDIANS.get(col, 0))
X[CAT_COLS] = enc.transform(X[CAT_COLS]).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')
print(f'Features: {FEATURE_COLS}')

Train: 8,000 | Test: 2,000
Features: ['dep_hour', 'arr_hour', 'month', 'dayofweek', 'distance', 'temp', 'wspd', 'prcp', 'snow', 'coco', 'origin_topK', 'dest_topK', 'carrier', 'route_avg_delay']


In [4]:
# Train all three models
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=500, num_leaves=63, learning_rate=0.05,
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1, verbose=-1
    ),
}

results = {}
for name, model in models.items():
    print(f'Training {name}...', end=' ')
    # LR needs float features, not Categorical
    Xtr = X_train.copy().astype(float) if name == 'Logistic Regression' else X_train.copy()
    Xte = X_test.copy().astype(float)  if name == 'Logistic Regression' else X_test.copy()
    if name == 'LightGBM':
        for col in CAT_COLS:
            Xtr[col] = pd.Categorical(Xtr[col])
            Xte[col] = pd.Categorical(Xte[col])
    model.fit(Xtr, y_train)
    y_pred  = model.predict(Xte)
    y_proba = model.predict_proba(Xte)
    acc     = accuracy_score(y_test, y_pred)
    # One-vs-rest ROC-AUC
    y_bin   = label_binarize(y_test, classes=[0,1,2])
    roc_auc = roc_auc_score(y_bin, y_proba, multi_class='ovr', average='macro')
    results[name] = {
        'model':   model,
        'y_pred':  y_pred,
        'y_proba': y_proba,
        'acc':     acc,
        'roc_auc': roc_auc,
    }
    print(f'Accuracy={acc:.3f}  ROC-AUC={roc_auc:.3f}')

Training Logistic Regression... 

Accuracy=0.411  ROC-AUC=0.557
Training Random Forest... 

Accuracy=0.715  ROC-AUC=0.569
Training LightGBM... 

Accuracy=0.665  ROC-AUC=0.600


In [5]:
# Classification reports
for name, res in results.items():
    print(f'\n══ {name} ══')
    print(classification_report(y_test, res['y_pred'],
          target_names=['On-time','Minor Delay','Major Delay']))


══ Logistic Regression ══
              precision    recall  f1-score   support

     On-time       0.84      0.43      0.57      1629
 Minor Delay       0.10      0.21      0.14       206
 Major Delay       0.11      0.48      0.18       165

    accuracy                           0.41      2000
   macro avg       0.35      0.37      0.29      2000
weighted avg       0.71      0.41      0.49      2000


══ Random Forest ══
              precision    recall  f1-score   support

     On-time       0.82      0.86      0.84      1629
 Minor Delay       0.12      0.09      0.10       206
 Major Delay       0.09      0.07      0.08       165

    accuracy                           0.71      2000
   macro avg       0.34      0.34      0.34      2000
weighted avg       0.69      0.71      0.70      2000


══ LightGBM ══
              precision    recall  f1-score   support

     On-time       0.84      0.77      0.81      1629
 Minor Delay       0.14      0.18      0.16       206
 Major Dela

In [6]:
# Summary comparison table
from sklearn.metrics import f1_score

rows = []
for name, res in results.items():
    rows.append({
        'Model':        name,
        'Accuracy':     f"{res['acc']*100:.1f}%",
        'Macro F1':     f"{f1_score(y_test, res['y_pred'], average='macro'):.3f}",
        'Weighted F1':  f"{f1_score(y_test, res['y_pred'], average='weighted'):.3f}",
        'ROC-AUC (OvR)': f"{res['roc_auc']:.3f}",
    })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

              Model Accuracy Macro F1 Weighted F1 ROC-AUC (OvR)
Logistic Regression    41.1%    0.294       0.492         0.557
      Random Forest    71.5%    0.339       0.700         0.569
           LightGBM    66.5%    0.374       0.687         0.600


In [ ]:
# Bar chart — Accuracy & ROC-AUC side by side
names    = list(results.keys())
accs     = [results[n]['acc']     for n in names]
roc_aucs = [results[n]['roc_auc'] for n in names]
colors   = ['#6366f1','#f59e0b','#10b981']

x = np.arange(len(names))
w = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - w/2, accs,     w, label='Accuracy',   color=colors, alpha=0.85)
bars2 = ax.bar(x + w/2, roc_aucs, w, label='ROC-AUC',    color=colors, alpha=0.45, hatch='//')

ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=11)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('Model Comparison — Accuracy & ROC-AUC (macro OvR)', fontsize=13)
ax.legend()
ax.axhline(0.58, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)

for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('../assets/model_comparison_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: model_comparison_bar.png')

In [ ]:
# ROC curves — one plot per class
CLASS_NAMES = ['On-time', 'Minor Delay', 'Major Delay']
MODEL_COLORS = {'Logistic Regression': '#6366f1', 'Random Forest': '#f59e0b', 'LightGBM': '#10b981'}
y_bin = label_binarize(y_test, classes=[0,1,2])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for cls_idx, (ax, cls_name) in enumerate(zip(axes, CLASS_NAMES)):
    for name, res in results.items():
        fpr, tpr, _ = roc_curve(y_bin[:, cls_idx], res['y_proba'][:, cls_idx])
        roc = auc(fpr, tpr)
        ax.plot(fpr, tpr, label=f'{name} (AUC={roc:.3f})', color=MODEL_COLORS[name], linewidth=2)
    ax.plot([0,1],[0,1],'k--', linewidth=0.8, alpha=0.5)
    ax.set_title(f'ROC — {cls_name}', fontsize=11)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.legend(fontsize=8)
    ax.set_xlim([0,1]); ax.set_ylim([0,1.02])

plt.suptitle('ROC Curves by Delay Class', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../assets/model_comparison_roc.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: model_comparison_roc.png')

In [ ]:
# LightGBM feature importance (for thesis discussion)
lgbm_model = results['LightGBM']['model']
imp = pd.Series(lgbm_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
imp.plot(kind='barh', ax=ax, color='#10b981', alpha=0.85)
ax.set_title('LightGBM Feature Importance', fontsize=12)
ax.set_xlabel('Importance score')
plt.tight_layout()
plt.savefig('../assets/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: feature_importance.png')